# 🏗️ Modul 04 Hands-on: The Production PyTorch Pipeline

Di hands-on ini, Anda akan membangun pipeline Deep Learning berstandar industri dari hulu ke hilir:
1. **Custom Dataset & DataLoader**: Mengemas data tabular riil ke dalam batch pipeline efisien.
2. **Modular Architecture**: Menulis `nn.Module` yang bersih dan terstruktur.
3. **Modular Train & Evaluation Engine**: Memisahkan logika epoch, tracking metrik, dan validasi.
4. **Early Stopping & Model Checkpointing**: Menyimpan state dict model terbaik secara otomatis.
5. **Production Inference**: Me-load kembali bobot model tersimpan untuk memprediksi data pelanggan baru.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import os

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Pipeline berjalan di device: {device}")

## 1. Menyiapkan Dataset Realistis: Customer Churn Tabular
Kita simulasikan 2.000 data pelanggan telecom dengan 6 fitur:
`[Age, Tenure_Months, Monthly_Charges, Total_Spend, Support_Tickets, Online_Services]`

In [ ]:
np.random.seed(42)
N = 2000

age = np.random.randint(18, 70, N)
tenure = np.random.randint(1, 72, N)
monthly = np.random.uniform(20.0, 120.0, N)
spend = monthly * tenure + np.random.normal(0, 50, N)
tickets = np.random.poisson(lam=1.5, size=N)
services = np.random.randint(0, 5, N)

# Rumus probabilitas churn realistis: tiket banyak & biaya bulanan tinggi memicu churn
churn_prob = 1 / (1 + np.exp(-(0.03*monthly + 0.6*tickets - 0.05*tenure - 1.5)))
churn = (np.random.rand(N) < churn_prob).astype(int)

df = pd.DataFrame({
    "Age": age, "Tenure": tenure, "MonthlyCharges": monthly,
    "TotalSpend": spend, "SupportTickets": tickets, "Services": services,
    "Churn": churn
})
print(f"Total data: {len(df)} baris | Distribusi Churn:\n{df["Churn"].value_counts(normalize=True)}")
df.head()

## 2. Membangun Custom PyTorch Dataset & DataLoader
Kita bagi data menjadi Train (80%) dan Validation (20%), lalu normalisasi fiturnya dengan `StandardScaler`.

In [ ]:
X = df.drop(columns=["Churn"]).values
y = df["Churn"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standarisasi fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = TabularDataset(X_train_scaled, y_train)
val_dataset = TabularDataset(X_val_scaled, y_val)

# DataLoader dengan batch_size 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

sample_x, sample_y = next(iter(train_loader))
print(f"Batch X shape: {sample_x.shape} -> [Batch_Size, Features]")
print(f"Batch y shape: {sample_y.shape} -> [Batch_Size, 1]")

## 3. Arsitektur Model: Modular ChurnClassifier

In [ ]:
class ChurnClassifier(nn.Module):
    def __init__(self, in_features=6, hidden_dim=32, dropout_rate=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)  # Output logit
        )
        
    def forward(self, x):
        return self.network(x)

model = ChurnClassifier().to(device)
print(model)

## 4. Modular Engine: Train Epoch, Evaluate & Early Stopping
Kita rancang fungsi training loop modular berstandar produksi yang menyimpan checkpoint terbaik ke file `best_churn_model.pt`.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
        
    return total_loss / total, correct / total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * len(y_batch)
            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader.dataset)
    preds_binary = (np.array(all_preds) > 0.5).astype(int)
    accuracy = (preds_binary == np.array(all_targets)).mean()
    auc = roc_auc_score(all_targets, all_preds)
    return avg_loss, accuracy, auc

## 5. Menjalankan Training dengan Early Stopping & Checkpointing

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

CHECKPOINT_PATH = "best_churn_model.pt"
best_val_loss = float("inf")
patience = 8
patience_counter = 0

print("🚀 Memulai Training Pipeline...")
for epoch in range(1, 61):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    
    # Checkpointing Logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        patience_counter = 0
        saved_flag = "💾 [CHECKPOINT SAVED]"
    else:
        patience_counter += 1
        saved_flag = ""
        
    if epoch % 5 == 0 or saved_flag:
        print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | Val Loss: {val_loss:.4f} AUC: {val_auc:.3f} {saved_flag}")
        
    if patience_counter >= patience:
        print(f"\n⏹️ Early Stopping terpicu pada epoch {epoch}: Validation loss tidak membaik selama {patience} epoch berturut-turut.")
        break

## 6. Production Inference: Me-load Model Tersimpan untuk Prediksi Pelanggan Baru
Simulasikan data seorang pelanggan baru yang baru saja masuk ke sistem:
`Age=45, Tenure=3, MonthlyCharges=110.0, TotalSpend=330.0, SupportTickets=4, Services=1`

In [ ]:
# 1. Inisialisasi model kosong dan load state_dict dari file checkpoint
deployed_model = ChurnClassifier().to(device)
deployed_model.load_state_dict(torch.load(CHECKPOINT_PATH))
deployed_model.eval()
print("✅ Berhasil memuat bobot model dari checkpoint file!")

# 2. Input data pelanggan baru
new_customer = np.array([[45, 3, 110.0, 330.0, 4, 1]])
new_customer_scaled = scaler.transform(new_customer)  # Gunakan scaler yang sama!
new_customer_tensor = torch.tensor(new_customer_scaled, dtype=torch.float32).to(device)

# 3. Inferensi
with torch.no_grad():
    logit_output = deployed_model(new_customer_tensor)
    churn_probability = torch.sigmoid(logit_output).item()

print(f"\n📊 HASIL INFERENSI PRODUKSI:")
print(f"Probabilitas Churn: {churn_probability * 100:.2f}%")
if churn_probability > 0.5:
    print("⚠️ STATUS: BERISIKO TINGGI CHURN (Kirim promo retensi segera!)")
else:
    print("🟢 STATUS: PELANGGAN SETIA (Risiko rendah)")